In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Pag-initialize ng Qubit

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago pa.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Kapag nag-execute ng Circuit sa isang IBM&reg; quantum processing unit (QPU), karaniwang may implicit reset na idiniinsert sa simula ng Circuit para matiyak na naininitialize sa zero ang mga Qubit. Kinokontrol ito ng `init_qubits` flag, na itinakda bilang isang [primitive execution option](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2).

Gayunpaman, hindi perpekto ang proseso ng reset, kaya nagdudulot ito ng mga state preparation error. Para mabawasan ang error, naglalagay din ang system ng repetition delay time (o `rep_delay`) sa pagitan ng mga Circuit. Ang bawat backend ay may iba-ibang default na `rep_delay`, pero karaniwang mas matagal ito kaysa sa T1 para makapag-reset ang environment ng mga Qubit. Maaaring i-query ang default na `rep_delay` sa pamamagitan ng pagpapatakbo ng `backend.default_rep_delay`.

Lahat ng IBM QPU ay gumagamit ng dynamic repetition rate execution, na nagbibigay-daan sa pagbabago ng `rep_delay` para sa bawat job. Ang mga Circuit na isinusumite mo sa isang primitive job ay pinagsama-sama para sa execution sa QPU. Ang mga Circuit na ito ay ine-execute sa pamamagitan ng pag-ulit-ulit sa mga Circuit para sa bawat shot na hinihingi; ang execution ay column-wise sa isang matrix ng mga Circuit at shot, gaya ng makikita sa sumusunod na larawan.

![Ang unang column ay kumakatawan sa shot 0. Ang mga Circuit ay pinapatakbo nang sunud-sunod mula 0 hanggang 3. Ang pangalawang column ay kumakatawan sa shot 1. Ang mga Circuit ay pinapatakbo nang sunud-sunod mula 0 hanggang 3. Ang natitirang mga column ay sumusunod sa parehong pattern.](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "Column-wise execution matrix")

Dahil ang `rep_delay` ay idiniinsert sa pagitan ng mga Circuit, ang bawat shot ng execution ay nararamdaman ang delay na ito. Kaya naman, ang pagbabago ng `rep_delay` sa mas mababang halaga ay nagpapababa ng kabuuang oras ng QPU execution, ngunit sa gastos ng mas mataas na rate ng state preparation error, gaya ng makikita sa sumusunod na larawan:

![Ipinapakita ng larawang ito na habang bumababa ang halaga ng `rep_delay`, tumataas ang rate ng state preparation error.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "Repetition delay versus error rate")

Ang pagtakda ng parehong `rep_delay=0` at `init_qubits=False` ay mahalagang "pinagsasama" ang mga Circuit, dahil magsisimula ang mga Qubit sa huling state mula sa nakaraang shot.

Tandaan na kahit ang mga Circuit sa isang primitive job ay pinagsama-sama para sa QPU execution, walang garantiya sa pagkakasunud-sunod ng pag-execute ng mga Circuit mula sa mga PUB. Kaya naman, kahit isumite mo ang `pubs=[pub1, pub2]`, walang garantiya na mauuna ang mga Circuit mula sa `pub1` bago ang mula sa `pub2`. Wala ring garantiya na ang mga Circuit mula sa parehong job ay tatakbo bilang isang batch sa QPU.

## Pagtukoy ng rep_delay para sa isang primitive job

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## Mga susunod na hakbang
> **Tip:** - Subukan ang isang halimbawa sa tutorial na [Quantum approximate optimization algorithm (QAOA)](/tutorials/quantum-approximate-optimization-algorithm).
>   - Suriin kung paano [magsimula sa mga primitive.](./get-started-with-primitives)